# Task 6 — Idempotent Replay Verification

The canonical Stage 3 workflow rebuilds the five-file baseline, restarts Spark with the existing checkpoint, modifies only `src/datasets/__init__.py`, replays that file with a new `run_id`, waits for Kafka Connect and Spark, removes stale graph entities, and validates the final stores. The cells below consume the strict manifest created by `bash scripts/finalize_stage3_evidence.sh`; they never connect to live services during a Pages build.


In [ ]:
from pathlib import Path
import json

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'screenshots/replay/stage3_replay_manifest.json').exists())
MANIFEST_PATH = ROOT / 'screenshots/replay/stage3_replay_manifest.json'
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
assert manifest['stage'] == 3 and manifest['status'] == 'pass'
print('manifest:', MANIFEST_PATH.relative_to(ROOT))
print('captured_at:', manifest['run']['captured_at'])


## Modified file and run identity


In [ ]:
source = manifest['source']
print('target:', source['target_file'])
print('file_id:', source['file_id'])
print('content_hash:', source['before_content_hash'], '->', source['after_content_hash'])
print('run_id:', manifest['run']['baseline_run_id'], '->', manifest['run']['replay_run_id'])
assert source['before_content_hash'] != source['after_content_hash']
assert manifest['run']['baseline_run_id'] != manifest['run']['replay_run_id']
assert source['source_restored'] is True


## Kafka and Spark checkpoint proof


In [ ]:
print('Kafka replay delta:', manifest['kafka']['delta'])
print('Spark metadata offsets:', manifest['spark']['metadata_offset_before'], '->', manifest['spark']['metadata_offset_after_restart'], '->', manifest['spark']['metadata_offset_after_replay'])
assert manifest['kafka']['delta'] == {'cpg.nodes': 23, 'cpg.edges': 16, 'cpg.metadata': 1, 'cpg.errors': 0}
assert [manifest['spark']['metadata_offset_before'], manifest['spark']['metadata_offset_after_restart'], manifest['spark']['metadata_offset_after_replay']] == [5, 5, 6]


## Neo4j cleanup and MongoDB replacement


In [ ]:
neo4j = manifest['neo4j']
mongo = manifest['mongodb']
print('Neo4j explicit nodes:', neo4j['before']['explicit_nodes'], '->', neo4j['pre_cleanup']['explicit_nodes'], '->', neo4j['after']['explicit_nodes'])
print('Neo4j relationships:', neo4j['before']['edges'], '->', neo4j['pre_cleanup']['edges'], '->', neo4j['after']['edges'])
print('stale deleted:', neo4j['stale_deleted'])
print('MongoDB documents:', mongo['documents_before'], '->', mongo['documents_after_replay'], '; unchanged:', mongo['unchanged_documents'])
assert neo4j['stale_deleted'] == {'nodes': 3, 'edges': 2}
assert neo4j['after']['duplicate_node_groups'] == neo4j['after']['duplicate_edge_groups'] == 0
assert mongo['documents_after_replay'] == 5 and mongo['unchanged_documents'] == 4


## Database UI evidence

Neo4j Browser after stale cleanup:

![Neo4j replay state](../screenshots/replay/neo4j_after_cleanup.png)

Mongo Express after metadata replacement:

![MongoDB replay document](../screenshots/replay/mongodb_after_replay.png)


## Reflection

**What worked:** deterministic file-level replay, Kafka Connect `MERGE`, MongoDB replacement by `file_id`, and the persisted Spark checkpoint form one auditable idempotency chain. **What failed during design review:** the original script selected `config.py`, which was absent from the five-file baseline, and treated expected Kafka replay events as invalid duplicate IDs. **Resolution:** the canonical target is now a baseline file, Kafka append-log repetition is measured separately from store duplication, and cleanup runs only after connector lag reaches zero. **Limitation:** structural AST IDs may change after edits, so correctness still depends on file-scoped stale cleanup rather than a production-grade semantic diff.
